In [1]:
import pandera.pandas as pa
import pandas as pd
import numpy as np
import pyarrow
from pathlib import Path

In [2]:
def LengthCheck(size: int) -> pa.Check:
    return pa.Check(lambda x: x.apply(len) == size, error=f'List must have length {size}')


data_datasframe_schema = pa.DataFrameSchema({
    'id': pa.Column(
        pd.ArrowDtype(pyarrow.uint64()),
        unique=True,
        checks=[
            pa.Check.ge(0),
        ]),
    'target': pa.Column(
        pd.ArrowDtype(pyarrow.uint8()),
        checks=[
            pa.Check.ge(0),
            pa.Check.le(1),
        ]),
    'el_lhmedium': pa.Column(
        pd.ArrowDtype(pyarrow.bool_()),
    ),
    'el_lhvloose': pa.Column(
        pd.ArrowDtype(pyarrow.bool_()),
    ),
    'trig_L2_calo_eta': pa.Column(
        pd.ArrowDtype(pyarrow.float32()),
        checks=[
            pa.Check.ge(-2.5),
            pa.Check.le(2.5),
        ]),
    'trig_L2_calo_et': pa.Column(
        pd.ArrowDtype(pyarrow.float32()),
        checks=[
            pa.Check.ge(0),
            pa.Check.le(1e12),
        ]),
    'trig_L2_calo_rings': pa.Column(
        pd.ArrowDtype(pyarrow.list_(pyarrow.float32())),
    ),
})

In [3]:
def generate_example(n_rows: int, id_start: int = 0) -> pd.DataFrame:
    data = {
    'id': np.arange(id_start, id_start + n_rows, dtype=np.uint64),
    'target': np.random.randint(0, 2, n_rows, dtype=np.uint8),
    'el_lhmedium': np.random.choice([True, False], n_rows),
    'el_lhvloose': np.random.choice([True, False], n_rows),
    'trig_L2_calo_eta': np.random.uniform(-2.5, 2.5, n_rows).astype(np.float32),
    'trig_L2_calo_et': np.random.uniform(12e3, 100e3, n_rows).astype(np.float32),
    'trig_L2_calo_rings': [np.random.uniform(0, 1e6, 100).astype(np.float32) for _ in range(n_rows)]
    }
    dtype = pyarrow.schema({
        'id': pyarrow.uint64(),
        'target': pyarrow.uint8(),
        'el_lhmedium': pyarrow.bool_(),
        'el_lhvloose': pyarrow.bool_(),
        'trig_L2_calo_eta': pyarrow.float32(),
        'trig_L2_calo_et': pyarrow.float32(),
        'trig_L2_calo_rings': pyarrow.list_(pyarrow.float32())
    })
    table = pyarrow.Table.from_pydict(data, schema=dtype)
    example_df = table.to_pandas(types_mapper=pd.ArrowDtype)
    example_df = data_datasframe_schema.validate(example_df)
    return example_df

In [4]:
output_dir = Path.home() / 'workspaces' / 'ringer-zero' / 'tests' / 'data' / 'test_dataset' / 'data.parquet'
output_dir.mkdir(parents=True, exist_ok=True)

In [5]:
# Generate example data with PyArrow dtypes
n_rows = 100000
for i in range(2):
    example_df = generate_example(n_rows, id_start=i*n_rows)
    example_df.to_parquet(
        f'{str(Path.home())}/workspaces/ringer-zero/tests/data/test_dataset/data.parquet/data_{i:04d}.parquet')
example_df

,id,target,el_lhmedium,el_lhvloose,trig_L2_calo_eta,trig_L2_calo_et,trig_L2_calo_rings
0,100000,1,False,True,2.332784,71039.140625,[354691.16 607679.3 840957.25 918335.9 ...
1,100001,0,True,False,-0.900391,96277.679688,[110529.96 703863.06 16123.603 702791.9 9...
2,100002,1,False,False,-1.722807,24871.955078,[497711.28 235140.98 797080.5 452939.34 9...
3,100003,0,True,False,-0.654307,96276.328125,[201603.1 248513.31 200742.95 842514.25...
4,100004,1,True,False,-0.278427,72915.0,[ 45845.008 455637.56 558759.5 400118.22 4...
...,...,...,...,...,...,...,...
99995,199995,0,False,True,-1.838954,55049.308594,[5.07025344e+05 4.60397781e+05 9.49099219e+04 ...
99996,199996,1,True,False,0.13128,47002.984375,[137036.97 706352.25 763111. 86049.26 8...
99997,199997,0,False,True,0.549191,32605.550781,[616331.06 446422.6 635919.25 232398.02...
99998,199998,0,False,False,-1.389722,93270.929688,[974601.06 339223.9 560298.6 258966.39 7...


In [6]:
import duckdb
with duckdb.connect(':memory:') as conn:
    query = "SELECT * FROM parquet_scan('/home/lucasbanunes/workspaces/ringer-zero/tests/data/test_dataset/data.parquet/*.parquet')"
    df = conn.execute(query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype                  
---  ------              --------------   -----                  
 0   id                  200000 non-null  uint64[pyarrow]        
 1   target              200000 non-null  uint8[pyarrow]         
 2   el_lhmedium         200000 non-null  bool[pyarrow]          
 3   el_lhvloose         200000 non-null  bool[pyarrow]          
 4   trig_L2_calo_eta    200000 non-null  float[pyarrow]         
 5   trig_L2_calo_et     200000 non-null  float[pyarrow]         
 6   trig_L2_calo_rings  200000 non-null  list<l: float>[pyarrow]
dtypes: bool[pyarrow](2), float[pyarrow](2), list<l: float>[pyarrow](1), uint64[pyarrow](1), uint8[pyarrow](1)
memory usage: 80.5 MB


In [7]:
data_datasframe_schema.validate(df)

,id,target,el_lhmedium,el_lhvloose,trig_L2_calo_eta,trig_L2_calo_et,trig_L2_calo_rings
0,0,0,True,False,1.04938,29434.910156,[267025.12 171002.34 49545.312 460042.94 6...
1,1,0,True,False,-2.359512,14700.324219,[312402.62 430681.1 518448.88 500855.53 3...
2,2,1,False,True,-1.96512,85653.367188,[4.06918750e+05 7.68142875e+05 4.74786906e+05 ...
3,3,0,False,True,0.826057,12306.545898,[1.50693750e+05 7.31558984e+04 7.85146625e+05 ...
4,4,0,False,True,2.0738,45551.382812,[620647.6 447834. 842371.3 438302.97 2...
...,...,...,...,...,...,...,...
199995,199995,0,False,True,-1.838954,55049.308594,[5.07025344e+05 4.60397781e+05 9.49099219e+04 ...
199996,199996,1,True,False,0.13128,47002.984375,[137036.97 706352.25 763111. 86049.26 8...
199997,199997,0,False,True,0.549191,32605.550781,[616331.06 446422.6 635919.25 232398.02...
199998,199998,0,False,False,-1.389722,93270.929688,[974601.06 339223.9 560298.6 258966.39 7...
